### Packages

In [ ]:
library(tidyverse)
library(lme4)
library(lmerTest)
library(broom)
library(broom.mixed)
library(ggeffects)
library(emmeans)
library(performance)
library(patchwork)
library(cowplot)
library(modelsummary)
library(datawizard)
# source("scripts/simulate_lmm_data.R")

In [ ]:
simulate_lmm_data <- function() {
 
set.seed(123)
 
n_subjects <- 30
n_time <- 6
 
subject_df <- tibble(
  subject = factor(1:n_subjects),
  b0 = rnorm(n_subjects, mean = 0, sd = 6),
  b1 = rnorm(n_subjects, mean = 0, sd = 1.2)
)
 
df <- expand_grid(
  subject = factor(1:n_subjects),
  time = 0:(n_time - 1)
) |>
  left_join(subject_df, by = "subject") |>
  mutate(
    y = 20 + 2.5 * time + b0 + b1 * time + rnorm(n(), mean = 0, sd = 2)
  )
  return(df)
  }

In [ ]:
tbl1 <- read.csv("050_ADNI_Merge_Cohort_V3")

# Biomarkers Over Time (w/out IR)

## ABETA

In [ ]:
m1_a <- lmer(
    ABETA_capped ~ VISCODEnum_z + (VISCODEnum_z | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m1_a)
confint(m1_a)

In [ ]:
# CSF ABETA Significantly decreases over time (suggesting progression)

In [ ]:
tbl1 <- tbl1 |>
    mutate(pred_m1_a = predict(m1_a))
# Graphing m1_a w/ Random slope and intercept
m1_a_graph <- ggplot(tbl1, aes(VISCODEnum_z, ABETA_capped, colour = RID)) +
    geom_point(alpha = 0.5) +
    geom_line(aes(y = pred_m1_a, group = RID), linewidth = 0.9) +
    guides(colour = "none") +
    labs(
        title = "Mixed model with random intercept and slope: ABETA trajectory over time",
        subtitle = "Different subject baselines and slopes",
        x = "Months since baseline (scaled)",
        y = "CSF ABETA"
      ) +
    theme_minimal(base_size = 13) 
m1_a_graph

In [ ]:
# Population predicted relationship
m1_a_pred <- ggpredict(m1_a, terms = "VISCODEnum_z") |>
  plot() +
  labs(
    title = "",
    x = "SD of Months Since Baseline",
    y = "CSF AB"
  ) +
  theme_minimal(base_size = 13)
m1_a_pred

## TAU

In [ ]:
m1_t <- lmer(
    TAU_capped ~ VISCODEnum_z + (VISCODEnum_z | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m1_t)

In [ ]:
# CSF TAU Significantly increases over time (suggesting progression)

In [ ]:
tbl1 <- tbl1 |>
    mutate(pred_m1_t = predict(m1_t))
# Graphing m1_a w/ Random slope and intercept
m1_a_graph <- ggplot(tbl1, aes(VISCODEnum_z, TAU_capped, colour = RID)) +
    geom_point(alpha = 0.5) +
    geom_line(aes(y = pred_m1_t, group = RID), linewidth = 0.9) +
    guides(colour = "none") +
    labs(
        title = "Mixed model with random intercept and slope: TAU trajectory over time",
        subtitle = "Different subject baselines and slopes",
        x = "Months since baseline (scaled)",
        y = "CSF TAU"
      ) +
    theme_minimal(base_size = 13) 
m1_a_graph

In [ ]:
# Population predicted relationship
m1_t_pred <- ggpredict(m1_t, terms = "VISCODEnum_z") |>
  plot() +
  labs(
    title = "",
    x = "SD of Months Since Baseline",
    y = "CSF t-Tau"
  ) +
  theme_minimal(base_size = 13)
m1_t_pred

## PTAU

In [ ]:
m1_p <- lmer(
    PTAU_capped ~ VISCODEnum_z + (VISCODEnum_z | RID),
    data = tbl1,
    control = lmerControl(optimizer = "bobyqa"))
summary(m1_p)

In [ ]:
tbl1 <- tbl1 |>
    mutate(pred_m1_p = predict(m1_p))
# Population predicted relationship
m1_p_pred <- ggpredict(m1_p, terms = "VISCODEnum_z") |>
  plot() +
  labs(
    title = "",
    x = "SD of Months Since Baseline",
    y = "CSF p-Tau"
  ) +
  theme_minimal(base_size = 13)
m1_p_pred

## Comparisons

In [ ]:
# Model Summary Comparisons
modelsummary(
  list(
    "AB" = m1_a,
    "TAU" = m1_t,
    "PTAU" = m1_p
  ),
  statistic = "({std.error})")

In [ ]:
plot1 <- cowplot::plot_grid(
    m1_a_pred,
    m1_t_pred,
    m1_p_pred,
    labels = c("A", "B", "C"),
      nrow = 1,
      align = "hv")
plot1

In [ ]:
ggsave("065_Biomarkers Over Time Plot.png", plot1, width = 10, height = 5)

# Unadjusted Modelling

## ABETA

### METS-IR

In [ ]:
# Raw METS-IR
a_m1<- lmer(ABETA_capped ~ VISCODEnum_z * METSIR_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(a_m1)

In [ ]:
confint(a_m1)

In [ ]:
# METS-IR Tertiles
a_m2<- lmer(ABETA_capped ~ VISCODEnum_z * METS_IR_Tertiles + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(a_m2)

In [ ]:
confint(a_m2)

### TYG

In [ ]:
a_m3 <- lmer(ABETA_capped ~ VISCODEnum_z * TYG_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(a_m3)

In [ ]:
confint(a_m3, "TYG_capped", level = 0.95)
confint(a_m3, "VISCODEnum_z:TYG_capped", level = 0.95)

In [ ]:
a_m4 <- lmer(ABETA_capped ~ VISCODEnum_z * TYG_Tertiles + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(a_m4)

In [ ]:
confint(a_m4)

### TYG-BMI

In [ ]:
a_m5 <- lmer(ABETA_capped ~ VISCODEnum_z * TYG_BMI_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(a_m5)

In [ ]:
confint(a_m5, "TYG_BMI_capped", level = 0.95)
confint(a_m5, "VISCODEnum_z:TYG_BMI_capped", level = 0.95)

In [ ]:
a_m6 <- lmer(ABETA_capped ~ VISCODEnum_z * TYG_BMI_Tertiles + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(a_m6)

In [ ]:
confint(a_m6)

### TG/HDL

In [ ]:
a_m7 <- lmer(ABETA_capped ~ VISCODEnum_z * TG_HDL_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(a_m7)

In [ ]:
confint(a_m7, "TG_HDL_capped", level = 0.95)
confint(a_m7, "VISCODEnum_z:TG_HDL_capped", level = 0.95)

In [ ]:
a_m8 <- lmer(ABETA_capped ~ VISCODEnum_z * TG_HDL_Tertiles + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(a_m8)

In [ ]:
confint(a_m8)

### Model Comparison

In [ ]:
# Raw IR Metrics Model Summary
modelsummary(
  list(
    "METS-IR" = a_m1,
    "TYG" = a_m3,
    "TYG-BMI" = a_m5,
    "TG/HDL" = a_m7
  ),
  statistic = "({std.error})"
)

## TAU

### METS-IR

In [ ]:
# Raw
t_m1 <- lmer(TAU_capped ~ VISCODEnum_z * METSIR_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(t_m1)

In [ ]:
confint(t_m1, "METSIR_capped", level = 0.95)
confint(t_m1, "VISCODEnum_z:METSIR_capped", level = 0.95)

In [ ]:
# Tertiles
t_m2 <- lmer(TAU_capped ~ VISCODEnum_z * METS_IR_Tertiles + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(t_m2)

In [ ]:
confint(t_m2)

### TYG

In [ ]:
# Raw
t_m3 <- lmer(TAU_capped ~ VISCODEnum_z * TYG_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(t_m3)

In [ ]:
confint(t_m3, "TYG_capped", level = 0.95)
confint(t_m3, "VISCODEnum_z:TYG_capped", level = 0.95)

In [ ]:
# Tertiles
t_m4 <- lmer(TAU_capped ~ VISCODEnum_z * TYG_Tertiles + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(t_m4)

In [ ]:
confint(t_m4)

### TYG-BMI

In [ ]:
# Raw
t_m5 <- lmer(TAU_capped ~ VISCODEnum_z * TYG_BMI_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(t_m5)

In [ ]:
confint(t_m5, "TYG_BMI_capped", level = 0.95)
confint(t_m5, "VISCODEnum_z:TYG_BMI_capped", level = 0.95)

In [ ]:
# tertiles
t_m6 <- lmer(TAU_capped ~ VISCODEnum_z * TYG_BMI_Tertiles + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(t_m6)

In [ ]:
confint(t_m6)

### TG/HDL

In [ ]:
# Raw
t_m7 <- lmer(TAU_capped ~ VISCODEnum_z * TG_HDL_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(t_m7)

In [ ]:
confint(t_m7, "TG_HDL_capped", level = 0.95)
confint(t_m7, "VISCODEnum_z:TG_HDL_capped", level = 0.95)

In [ ]:
# tertiles
t_m8 <- lmer(TAU_capped ~ VISCODEnum_z * TG_HDL_Tertiles + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(t_m8)

In [ ]:
confint(t_m8)

### Model Comparison

In [ ]:
# Raw IR Metrics Model Summary
modelsummary(
  list(
    "METS-IR" = t_m1,
    "TYG" = t_m3,
    "TYG-BMI" = t_m5,
    "TG/HDL" = t_m7
  ),
  statistic = "({std.error})"
)

## PTAU

### METS-IR

In [ ]:
# Raw
p_m1 <- lmer(PTAU_capped ~ VISCODEnum_z * METSIR_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(p_m1)

In [ ]:
confint(p_m1, "METSIR_capped", level = 0.95)
confint(p_m1, "VISCODEnum_z:METSIR_capped", level = 0.95)

In [ ]:
# Tertiles
p_m2 <- lmer(PTAU_capped ~ VISCODEnum_z * METS_IR_Tertiles + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(p_m2)

In [ ]:
confint(p_m2)

### TYG

In [ ]:
# Raw
p_m3 <- lmer(PTAU_capped ~ VISCODEnum_z * TYG_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(p_m3)

In [ ]:
confint(p_m3, "TYG_capped", level = 0.95)
confint(p_m3, "VISCODEnum_z:TYG_capped", level = 0.95)

In [ ]:
# Tertiles
p_m4 <- lmer(PTAU_capped ~ VISCODEnum_z * TYG_Tertiles + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(p_m4)

In [ ]:
confint(p_m4)

### TYG-BMI

In [ ]:
# Raw
p_m5 <- lmer(PTAU_capped ~ VISCODEnum_z * TYG_BMI_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(p_m5)

In [ ]:
confint(p_m5, "TYG_BMI_capped", level = 0.95)
confint(p_m5, "VISCODEnum_z:TYG_BMI_capped", level = 0.95)

In [ ]:
# Tertiles
p_m6 <- lmer(PTAU_capped ~ VISCODEnum_z * TYG_BMI_Tertiles + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(p_m6)

In [ ]:
confint(p_m6)

### TG/HDL

In [ ]:
# Raw
p_m7 <- lmer(PTAU_capped ~ VISCODEnum_z * TG_HDL_capped + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(p_m7)

In [ ]:
confint(p_m7, "TG_HDL_capped", level = 0.95)
confint(p_m7, "VISCODEnum_z:TG_HDL_capped", level = 0.95)

In [ ]:
# Tertiles
p_m8 <- lmer(PTAU_capped ~ VISCODEnum_z * TG_HDL_Tertiles + (VISCODEnum_z | RID), data = tbl1, control = lmerControl(optimizer = "bobyqa"))
summary(p_m8)

In [ ]:
confint(p_m8)

### Model Comparison

In [ ]:
# Raw IR Metrics Model Summary
modelsummary(
  list(
    "METS-IR" = p_m1,
    "TYG" = p_m3,
    "TYG-BMI" = p_m5,
    "TG/HDL" = p_m7
  ),
  statistic = "({std.error})"
)

## Graphing

### METS-IR

In [ ]:
tbl1 <- tbl1 |>
    mutate(pred_am2 = predict(a_m2),
           pred_tm2 = predict(t_m2),
           pred_pm2 = predict(p_m2))

In [ ]:
am2_g <- ggpredict(a_m2, terms = c("VISCODEnum_z", "METS_IR_Tertiles")) |>
    plot() +
    labs(
        title = "Predicted AB Over Time Stratified by METS-IR Tertile",
        x = "SD of Months Since Baseline",
        y = "Predicted AB (pg/mL)",
        color = "METS-IR Tertiles"
      ) +
  theme_minimal(base_size = 13)
am2_g

In [ ]:
tm2_g <- ggpredict(t_m2, terms = c("VISCODEnum_z", "METS_IR_Tertiles")) |>
    plot() +
    labs(
        title = "Predicted t-Tau Over Time Stratified by METS-IR Tertile",
        x = "SD of Months Since Baseline",
        y = "Predicted t-Tau (pg/mL)",
        color = "METS-IR Tertiles"
      ) +
  theme_minimal(base_size = 13)
tm2_g

In [ ]:
pm2_g <- ggpredict(p_m2, terms = c("VISCODEnum_z", "METS_IR_Tertiles")) |>
    plot() +
    labs(
        title = "Predicted p-Tau Over Time Stratified by METS-IR Tertile",
        x = "SD of Months Since Baseline",
        y = "Predicted p-Tau (pg/mL)",
        color = "METS-IR Tertiles"
      ) +
  theme_minimal(base_size = 13)
pm2_g

In [ ]:
ggsave("054_AB x Time x METS-IR Tertile.png", am2_g, width = 10, height = 5)
ggsave("055_TAU x Time x METS-IR Tertile.png", tm2_g, width = 10, height = 5)
ggsave("056_PTAU x Time x METS-IR Tertile.png", pm2_g, width = 10, height = 5)